# PlantEnzyme AI — классификация GH13 по ESM-2 эмбеддингам

Ноутбук консолидирует финальный пайплайн: 31 курированный белок (13 isoamylase / 12 alpha-amylase / 6 pullulanase),
homology-aware кросс-валидация (StratifiedGroupKFold по кластерам MMseqs2, seed=788), сравнение
Logistic Regression vs Linear SVM, и честный (без утечки гомологии) внешний тест.

**Происхождение данных и решения, зафиксированные до этого ноутбука:**
- Датасет собран из CAZy + UniProt/GenBank, вручную вычищен от фрагментов и близких дублей (>92% identity) — см. `docs/dataset_plan.md`.
- Кластеризация для CV: MMseqs2 `--min-seq-id 0.9 -c 0.8 --cov-mode 1` (порог 0.7 оказался слишком мягким — сливал весь класс активности в один кластер, см. историю экспериментов).
- Seed `788` для `StratifiedGroupKFold` найден перебором (10000 seeds) как первый, при котором все 5 фолдов содержат все 3 класса и кластеры не пересекаются между train/valid.
- Внешний тест: 6 кандидатов, найденных дополнительно; **один (`Q7X9T1`, 99.1% identity к тренировочному `P17859`) оказался утечкой** и в этом ноутбуке отфильтровывается автоматически по порогу идентичности, а не вручную.


## 0. Настройка

In [1]:
from pathlib import Path
import subprocess
import json as _json

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
)

pd.set_option("display.width", 120)


/Users/anastasia/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [12]:
# Пути. Ноутбук предполагается лежащим в notebooks/, т.е. PROJECT_ROOT — на уровень выше.
PROJECT_ROOT = Path(".").resolve()

DATASET_CSV = PROJECT_ROOT / "./data/processed/ml_dataset.csv"
FOLDS_CSV = PROJECT_ROOT / "./data/processed/folds_stratified.csv"

TRAIN_FASTA = PROJECT_ROOT / "./data/sequences/ml_dataset.fasta"

EXTERNAL_FASTA_RAW = PROJECT_ROOT / "./data/sequences/external_candidates.fasta"
EXTERNAL_CSV_RAW = PROJECT_ROOT / "./data/processed/external_candidates.csv"
EXTERNAL_MMSEQS_HITS = PROJECT_ROOT / "./data/processed/external_vs_training_mmseqs.csv"

MMSEQS_EXTERNAL_DIR = PROJECT_ROOT / "./data/processed/mmseqs_external"

TRAIN_EMB_NPY = PROJECT_ROOT / "./data/processed/esm2_t12_35m_residue_only_embeddings.npy"
TRAIN_EMB_META = PROJECT_ROOT / "./data/processed/esm2_t12_35m_residue_only_metadata.csv"

EXTERNAL_EMB_NPY = PROJECT_ROOT / "./data/processed/esm2_t12_35m_external_clean_embeddings.npy"
EXTERNAL_EMB_META = PROJECT_ROOT / "./data/processed/esm2_t12_35m_external_clean_metadata.csv"

MODEL_NAME = "facebook/esm2_t12_35M_UR50D"
BATCH_SIZE = 4
RANDOM_STATE = 788
N_SPLITS = 5

# Порог, при котором внешний кандидат считается утечкой (не независимым примером).
# Совпадает с порогом внутренней кластеризации train (0.9) для согласованности:
# если кандидат гомологичен train на уровне, где мы бы объединили белки в один
# кластер, он не может считаться независимым внешним примером.
LEAK_IDENTITY_THRESHOLD = 70.0  # см. обсуждение порога в разделе 4
LEAK_MIN_COVERAGE = 0.5

CLASSES = ["alpha-amylase", "isoamylase", "pullulanase/limit dextrinase"]

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"DEVICE = {DEVICE}")


PROJECT_ROOT = /Users/anastasia/work/pet-projects/plant-enzyme-ml
DEVICE = mps


## 1. Тренировочный датасет и homology-aware фолды

`ml_dataset.csv` — 31 курированная последовательность (см. историю чистки: фрагменты, аллельные дубликаты
и почти-идентичные ортологи вручную исключены). `folds_stratified.csv` — назначение fold для каждого
белка через `StratifiedGroupKFold(random_state=788)`, где группа = кластер гомологии MMseqs2
(`--min-seq-id 0.9`), так что ни один кластер не пересекает границу train/valid ни в одном fold.

In [13]:
dataset = pd.read_csv(DATASET_CSV)
folds = pd.read_csv(FOLDS_CSV)

metadata = dataset.merge(
    folds[["protein_id", "cluster_id", "fold"]],
    on="protein_id", how="left", validate="one_to_one",
)

assert metadata["fold"].notna().all()
assert metadata["cluster_id"].notna().all()
assert len(metadata) == 31
assert metadata["protein_id"].is_unique

print(f"Белков: {len(metadata)}")
print(f"Кластеров: {metadata['cluster_id'].nunique()}")
print("\nРаспределение классов:")
print(metadata["activity"].value_counts())

print("\nРаспределение по фолдам:")
print(pd.crosstab(metadata["fold"], metadata["activity"]))


Белков: 31
Кластеров: 29

Распределение классов:
activity
isoamylase                      13
alpha-amylase                   12
pullulanase/limit dextrinase     6
Name: count, dtype: int64

Распределение по фолдам:
activity  alpha-amylase  isoamylase  pullulanase/limit dextrinase
fold                                                             
0                     2           2                             1
1                     3           3                             1
2                     2           3                             1
3                     2           3                             2
4                     3           2                             1


## 2. ESM-2 эмбеддинги (residue-only mean pooling)

Берём последний скрытый слой, отбрасываем токены `<cls>`/`<eos>` и усредняем только по позициям
аминокислот. Эмбеддинги кэшируются в `.npy` — при повторном запуске пересчёт пропускается.

In [14]:
def embed_sequences(sequences: list[str], tokenizer, model, batch_size: int = BATCH_SIZE) -> np.ndarray:
    all_embeddings = []
    for start in range(0, len(sequences), batch_size):
        batch = sequences[start:start + batch_size]
        encoded = tokenizer(batch, return_tensors="pt", padding=True, truncation=False)
        input_ids = encoded["input_ids"].to(DEVICE)
        attention_mask = encoded["attention_mask"].to(DEVICE)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        hidden = outputs.last_hidden_state
        residue_hidden = hidden[:, 1:-1, :]
        residue_mask = attention_mask[:, 1:-1].unsqueeze(-1).to(residue_hidden.dtype)

        summed = (residue_hidden * residue_mask).sum(dim=1)
        counts = residue_mask.sum(dim=1)
        pooled = (summed / counts).detach().cpu().numpy()
        all_embeddings.append(pooled)

    return np.concatenate(all_embeddings, axis=0)


In [15]:
if TRAIN_EMB_NPY.exists() and TRAIN_EMB_META.exists():
    print("Загружаю кэшированные эмбеддинги train...")
    X_train = np.load(TRAIN_EMB_NPY)
    train_emb_meta = pd.read_csv(TRAIN_EMB_META)
    assert (train_emb_meta["protein_id"].values == metadata["protein_id"].values).all(), (
        "Порядок protein_id в кэше не совпадает с ml_dataset.csv — пересчитай эмбеддинги."
    )
else:
    print("Кэша нет — считаю эмбеддинги заново (нужен доступ к весам facebook/esm2_t12_35M_UR50D)...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()

    X_train = embed_sequences(metadata["sequence"].tolist(), tokenizer, model)
    np.save(TRAIN_EMB_NPY, X_train)
    metadata[["protein_id", "activity", "cluster_id", "fold", "length"]].to_csv(TRAIN_EMB_META, index=False)

assert X_train.shape == (31, 480)
assert not np.isnan(X_train).any() and not np.isinf(X_train).any()
print(f"X_train: {X_train.shape}")


Загружаю кэшированные эмбеддинги train...
X_train: (31, 480)


## 3. Кросс-валидация: Logistic Regression vs Linear SVM

Одна функция `run_cv` для обеих моделей вместо двух почти одинаковых скриптов.
Фолды берутся готовыми из `metadata['fold']` (уже посчитаны через `StratifiedGroupKFold`,
см. раздел 1) — здесь только обучение/оценка, без пересчёта сплита.

In [16]:
def run_cv(estimator, name: str) -> tuple[pd.DataFrame, np.ndarray, list, list]:
    y = metadata["activity"].values
    folds_arr = metadata["fold"].values

    results = []
    all_true, all_pred = [], []

    print("=" * 70)
    print(name)
    print("=" * 70)

    for fold in sorted(np.unique(folds_arr)):
        train_mask = folds_arr != fold
        valid_mask = folds_arr == fold

        estimator.fit(X_train[train_mask], y[train_mask])
        y_pred = estimator.predict(X_train[valid_mask])
        y_valid = y[valid_mask]

        acc = accuracy_score(y_valid, y_pred)
        f1 = f1_score(y_valid, y_pred, labels=CLASSES, average="macro", zero_division=0)
        prec = precision_score(y_valid, y_pred, labels=CLASSES, average="macro", zero_division=0)
        rec = recall_score(y_valid, y_pred, labels=CLASSES, average="macro", zero_division=0)

        results.append({
            "fold": fold, "n_train": train_mask.sum(), "n_valid": valid_mask.sum(),
            "accuracy": acc, "macro_precision": prec, "macro_recall": rec, "macro_f1": f1,
        })
        all_true.extend(y_valid)
        all_pred.extend(y_pred)

    results_df = pd.DataFrame(results)
    print(results_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
    print("\nMean ± std:")
    for metric in ["accuracy", "macro_precision", "macro_recall", "macro_f1"]:
        print(f"  {metric:20s} {results_df[metric].mean():.3f} ± {results_df[metric].std(ddof=1):.3f}")

    cm = confusion_matrix(all_true, all_pred, labels=CLASSES)
    cm_df = pd.DataFrame(cm, index=[f"true:{c}" for c in CLASSES], columns=[f"pred:{c}" for c in CLASSES])
    print("\nOut-of-fold confusion matrix:")
    print(cm_df)
    print("\nOut-of-fold classification report:")
    print(classification_report(all_true, all_pred, labels=CLASSES, zero_division=0, digits=3))

    return results_df, cm, all_true, all_pred


logistic = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE)),
])

svm = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LinearSVC(class_weight="balanced", random_state=RANDOM_STATE, max_iter=10000)),
])

logistic_cv_results, logistic_cm, logistic_true, logistic_pred = run_cv(logistic, "LOGISTIC REGRESSION — 5-fold CV")


LOGISTIC REGRESSION — 5-fold CV
 fold  n_train  n_valid  accuracy  macro_precision  macro_recall  macro_f1
    0       26        5     1.000            1.000         1.000     1.000
    1       24        7     1.000            1.000         1.000     1.000
    2       25        6     1.000            1.000         1.000     1.000
    3       24        7     1.000            1.000         1.000     1.000
    4       25        6     1.000            1.000         1.000     1.000

Mean ± std:
  accuracy             1.000 ± 0.000
  macro_precision      1.000 ± 0.000
  macro_recall         1.000 ± 0.000
  macro_f1             1.000 ± 0.000

Out-of-fold confusion matrix:
                                   pred:alpha-amylase  pred:isoamylase  pred:pullulanase/limit dextrinase
true:alpha-amylase                                 12                0                                  0
true:isoamylase                                     0               13                                  0
true:pul

In [17]:
svm_cv_results, svm_cm, svm_true, svm_pred = run_cv(svm, "LINEAR SVM — 5-fold CV")

LINEAR SVM — 5-fold CV
 fold  n_train  n_valid  accuracy  macro_precision  macro_recall  macro_f1
    0       26        5     1.000            1.000         1.000     1.000
    1       24        7     1.000            1.000         1.000     1.000
    2       25        6     1.000            1.000         1.000     1.000
    3       24        7     0.857            0.889         0.889     0.867
    4       25        6     1.000            1.000         1.000     1.000

Mean ± std:
  accuracy             0.971 ± 0.064
  macro_precision      0.978 ± 0.050
  macro_recall         0.978 ± 0.050
  macro_f1             0.973 ± 0.060

Out-of-fold confusion matrix:
                                   pred:alpha-amylase  pred:isoamylase  pred:pullulanase/limit dextrinase
true:alpha-amylase                                 12                0                                  0
true:isoamylase                                     0               12                                  1
true:pullulanase/

## 4. Честный внешний тест: автоматический фильтр утечки гомологии

Раньше отбор внешних кандидатов делался вручную по таблице MMseqs2 — и один кандидат
(`Q7X9T1`, 99.1% identity к тренировочному `P17859`) проскользнул. Здесь порог применяется
программно: любой кандидат с identity ≥ `LEAK_IDENTITY_THRESHOLD` к любому белку из train
(при достаточном coverage) исключается автоматически, без ручной проверки.

Если файла `external_vs_training_mmseqs.csv` ещё нет — нужно один раз посчитать его через MMseqs2
(команды ниже; требуют установленный `mmseqs` в PATH).

In [18]:
def run_mmseqs_external_vs_training():
    """Пересчитывает external_vs_training_mmseqs.csv, если его ещё нет."""
    MMSEQS_EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)
    query_db = MMSEQS_EXTERNAL_DIR / "external_db"
    target_db = MMSEQS_EXTERNAL_DIR / "training_db"
    result_db = MMSEQS_EXTERNAL_DIR / "external_vs_training"
    result_tsv = MMSEQS_EXTERNAL_DIR / "external_vs_training.tsv"

    def run(cmd):
        print("$ " + " ".join(map(str, cmd)))
        subprocess.run(cmd, check=True)

    run(["mmseqs", "createdb", str(EXTERNAL_FASTA_RAW), str(query_db)])
    run(["mmseqs", "createdb", str(TRAIN_FASTA), str(target_db)])
    run([
        "mmseqs", "search", str(query_db), str(target_db), str(result_db),
        str(MMSEQS_EXTERNAL_DIR / "tmp"),
        "--min-seq-id", "0.0", "-c", "0.0", "--cov-mode", "0", "-e", "0.001", "--threads", "4",
    ])
    run([
        "mmseqs", "convertalis", str(query_db), str(target_db), str(result_db), str(result_tsv),
        "--format-output", "query,target,pident,alnlen,qstart,qend,tstart,tend,evalue,bits",
    ])

    cols = ["query", "target", "pident", "alnlen", "qstart", "qend", "tstart", "tend", "evalue", "bits"]
    hits = pd.read_csv(result_tsv, sep="\t", names=cols)

    def fasta_lengths(path):
        lengths, cur_id, cur_len = {}, None, 0
        with open(path) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                if line.startswith(">"):
                    if cur_id is not None:
                        lengths[cur_id] = cur_len
                    cur_id = line[1:].split()[0]
                    cur_len = 0
                else:
                    cur_len += len(line)
            if cur_id is not None:
                lengths[cur_id] = cur_len
        return lengths

    qlen = fasta_lengths(EXTERNAL_FASTA_RAW)
    tlen = fasta_lengths(TRAIN_FASTA)

    best = (
        hits.sort_values(["query", "bits", "evalue"], ascending=[True, False, True])
        .groupby("query", as_index=False).first()
    )
    best["query_length"] = best["query"].map(qlen)
    best["target_length"] = best["target"].map(tlen)
    best["query_coverage"] = best["alnlen"] / best["query_length"]
    best["target_coverage"] = best["alnlen"] / best["target_length"]
    best.to_csv(EXTERNAL_MMSEQS_HITS, index=False)
    return best


if EXTERNAL_MMSEQS_HITS.exists():
    hits = pd.read_csv(EXTERNAL_MMSEQS_HITS)
else:
    hits = run_mmseqs_external_vs_training()

print(hits[["query", "target", "pident", "query_coverage", "target_coverage"]].to_string(index=False))


         query                                    target  pident  query_coverage  target_coverage
        A4PIS9 A0A0B6VSY7|isoamylase|Amaranthus_cruentus    37.3             NaN         0.869398
        Q7X9T1          P17859|alpha-amylase|Vigna_mungo    99.1             NaN         1.000000
        Q84UE6 A0A0B6VSY7|isoamylase|Amaranthus_cruentus    35.5             NaN         0.861716
        Q94A41          P17859|alpha-amylase|Vigna_mungo    54.3             NaN         0.942993
XP_004975057.1                  O81638|pullulanase/limit    84.6        1.004175         1.000000
XP_018715184.2          XP_021626046.1|pullulanase/limit    75.4        0.960042         0.945135


In [26]:
def read_fasta(path: Path) -> dict[str, str]:
    seqs, cur_id, cur_seq = {}, None, []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            if line.startswith(">"):
                if cur_id is not None:
                    seqs[cur_id] = "".join(cur_seq)
                #cur_id = line[1:].split("|")[0].split()[0]
                header = line[1:]
                parts = header.split("|")
                if len(parts) >= 2 and parts[0] in {"sp", "tr"}:
                    cur_id = parts[1]  # UniProt: db|ACCESSION|ENTRY_NAME ...
                else:
                    cur_id = parts[0].split()[0]  # кастомный 
                cur_seq = []
            else:
                cur_seq.append(line)
        if cur_id is not None:
            seqs[cur_id] = "".join(cur_seq)
    return seqs


external_seqs = read_fasta(EXTERNAL_FASTA_RAW)
external_meta_raw = pd.read_csv(EXTERNAL_CSV_RAW)

hits["coverage"] = hits[["query_coverage", "target_coverage"]].max(axis=1)
hits["is_leak"] = (hits["pident"] >= LEAK_IDENTITY_THRESHOLD) & (hits["coverage"] >= LEAK_MIN_COVERAGE)

print("Автоматическая фильтрация:")
for _, row in hits.iterrows():
    mark = "✗ ОТКЛОНЕНО" if row["is_leak"] else "✓ оставлен"
    print(f"  {mark:15s} {row['query']:16s} {row['pident']:5.1f}% identity, cov={row['coverage']:.2f} -> {row['target']}")

clean_ids = hits.loc[~hits["is_leak"], "query"].tolist()
print(f"\nВнешний тест ДО фильтра: {len(hits)}")
print(f"Внешний тест ПОСЛЕ фильтра (leakage-free): {len(clean_ids)}")

external_clean = external_meta_raw[external_meta_raw["accession"].isin(clean_ids)].reset_index(drop=True)
external_clean["sequence"] = external_clean["accession"].map(external_seqs)
assert external_clean["sequence"].notna().all()
external_clean


Автоматическая фильтрация:
  ✓ оставлен      A4PIS9            37.3% identity, cov=0.87 -> A0A0B6VSY7|isoamylase|Amaranthus_cruentus
  ✗ ОТКЛОНЕНО     Q7X9T1            99.1% identity, cov=1.00 -> P17859|alpha-amylase|Vigna_mungo
  ✓ оставлен      Q84UE6            35.5% identity, cov=0.86 -> A0A0B6VSY7|isoamylase|Amaranthus_cruentus
  ✓ оставлен      Q94A41            54.3% identity, cov=0.94 -> P17859|alpha-amylase|Vigna_mungo
  ✗ ОТКЛОНЕНО     XP_004975057.1    84.6% identity, cov=1.00 -> O81638|pullulanase/limit
  ✗ ОТКЛОНЕНО     XP_018715184.2    75.4% identity, cov=0.96 -> XP_021626046.1|pullulanase/limit

Внешний тест ДО фильтра: 6
Внешний тест ПОСЛЕ фильтра (leakage-free): 3


,accession,activity,organism,evidence_note,uniprot_header,length,download_status,error,sequence
0,Q84UE6,isoamylase,unknown,candidate external isoamylase,">tr|Q84UE6|Q84UE6_MAIZE Isoamylase 2, chloropl...",799.0,success,NaN,MASSLPAPPASPSSSWRGLTPRCPPPRCGPLLARAAARSYRYRFRT...
1,A4PIS9,isoamylase,unknown,candidate external isoamylase,>tr|A4PIS9|A4PIS9_PHAVU Isoamylase-type starch...,865.0,success,NaN,MASVVSAFSFSLIPFNQHKFGAKCVGFHKPLIQRKQRQLGLQNHFF...


## 5. Эмбеддинги для чистого внешнего набора и оценка

Финальные модели дообучаются на **всех** 31 тренировочных белках (не на fold-сплите — здесь
уже не кросс-валидация, а оценка на действительно независимых внешних примерах).

In [27]:
if EXTERNAL_EMB_NPY.exists() and EXTERNAL_EMB_META.exists():
    X_external = np.load(EXTERNAL_EMB_NPY)
    external_emb_meta = pd.read_csv(EXTERNAL_EMB_META)
    assert (external_emb_meta["accession"].values == external_clean["accession"].values).all()
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()
    X_external = embed_sequences(external_clean["sequence"].tolist(), tokenizer, model)
    np.save(EXTERNAL_EMB_NPY, X_external)
    external_clean[["accession", "activity", "organism", "evidence_note"]].to_csv(EXTERNAL_EMB_META, index=False)

print(f"X_external: {X_external.shape}")


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t12_35M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


X_external: (2, 480)


In [28]:
y_train_full = metadata["activity"].values
y_external_true = external_clean["activity"].values

logistic_final = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE)),
]).fit(X_train, y_train_full)

svm_final = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LinearSVC(class_weight="balanced", random_state=RANDOM_STATE, max_iter=10000)),
]).fit(X_train, y_train_full)

logistic_ext_pred = logistic_final.predict(X_external)
svm_ext_pred = svm_final.predict(X_external)

external_results = external_clean[["accession", "activity"]].copy()
external_results["logistic_pred"] = logistic_ext_pred
external_results["svm_pred"] = svm_ext_pred
external_results["logistic_correct"] = external_results["activity"] == external_results["logistic_pred"]
external_results["svm_correct"] = external_results["activity"] == external_results["svm_pred"]

print(external_results.to_string(index=False))

for name, preds in [("Logistic Regression", logistic_ext_pred), ("Linear SVM", svm_ext_pred)]:
    print(f"\n{name} — leakage-free external test:")
    print(f"  Accuracy: {accuracy_score(y_external_true, preds):.3f}")
    print(f"  Macro F1: {f1_score(y_external_true, preds, labels=CLASSES, average='macro', zero_division=0):.3f}")
    cm_ext = confusion_matrix(y_external_true, preds, labels=CLASSES)
    print(pd.DataFrame(cm_ext, index=[f"true:{c}" for c in CLASSES], columns=[f"pred:{c}" for c in CLASSES]))


accession   activity                logistic_pred                     svm_pred  logistic_correct  svm_correct
   Q84UE6 isoamylase pullulanase/limit dextrinase pullulanase/limit dextrinase             False        False
   A4PIS9 isoamylase pullulanase/limit dextrinase pullulanase/limit dextrinase             False        False

Logistic Regression — leakage-free external test:
  Accuracy: 0.000
  Macro F1: 0.000
                                   pred:alpha-amylase  pred:isoamylase  pred:pullulanase/limit dextrinase
true:alpha-amylase                                  0                0                                  0
true:isoamylase                                     0                0                                  2
true:pullulanase/limit dextrinase                   0                0                                  0

Linear SVM — leakage-free external test:
  Accuracy: 0.000
  Macro F1: 0.000
                                   pred:alpha-amylase  pred:isoamylase  pred:pull

## 6. Обсуждение: систематическая путаница isoamylase ↔ pullulanase

На независимых (низкогомологичных, <40% identity к train) изоамилазах обе модели ошибаются
**систематически**, и обе — в сторону pullulanase, а не alpha-amylase. Это не случайный шум:

- **Биологическое объяснение.** Isoamylase и pullulanase — оба debranching-фермента, режущие
  α-1,6-связи в точках ветвления крахмала/гликогена. Alpha-amylase режет α-1,4-связи внутри
  цепи — механистически другой тип реакции. Мы, судя по всему, наблюдаем, что модель выучила
  именно это структурное родство (debranching vs endo-hydrolysis), а не тонкую грань между
  isoamylase и pullulanase внутри debranching-группы.
- **Ограничение выборки.** Класс pullulanase — всего 6 примеров, isoamylase — 13, оба меньше,
  чем нужно для уверенного разделения близких по механизму классов линейным классификатором
  поверх усреднённых эмбеддингов.
- **Не техническая ошибка.** Это отличается от найденной ранее утечки (`Q7X9T1`, 99.1%
  identity) — там был баг в отборе кандидатов; здесь же — воспроизводимый паттерн ошибок
  модели на honest-данных, достойный отдельного абзаца Discussion, а не повод считать
  пайплайн сломанным.

**Возможные следующие шаги** (не реализованы в этом ноутбуке):
- Расширить обучающую выборку silver-labeled данными из UniProt (по Pfam-домену + EC), особенно
  для pullulanase/isoamylase — эту идею мы обсуждали в самом начале проекта.
- Попробовать нелинейный классификатор (MLP) поверх ESM-2-эмбеддингов — линейная граница может
  быть принципиально недостаточна для разделения двух debranching-подклассов.
- Явно смоделировать иерархию: сначала bracnhing-type (debranching vs non-debranching), затем
  тонкое разделение isoamylase/pullulanase внутри debranching-группы.